In [ ]:
# ============================================================================
# PARAMETERS — this cell is identical in all three notebooks.
# ============================================================================
RUN_MODE = "micro"          # "smoke" | "micro" (default) | "budget" | "full"
NUM_GPUS = None             # None = use every visible GPU; set 1 to force single-GPU
PUBLISH_KAGGLE_DATASET = True
CKPT_DATASET_SLUG = "dentex-repro-ckpts"
DATA_DATASET_SLUG = "dentex-repro-data"
REPO_URL = "https://github.com/AIscend-Research/dental-repro.git"

import os, subprocess, sys

# On Kaggle the repo is cloned into /kaggle/working (the only writable place
# that survives "Save Version"); locally the notebook already sits inside it.
if os.path.isdir("/kaggle/working"):
    CLONE = "/kaggle/working/repo"
    if os.path.isdir(os.path.join(CLONE, ".git")):
        subprocess.run(["git", "-C", CLONE, "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "50", REPO_URL, CLONE], check=True)
    PROJECT_ROOT = os.path.join(CLONE, "dentex-repro")
else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["RUN_MODE"] = RUN_MODE
print("project root:", PROJECT_ROOT)


In [ ]:
# ---- Environment: install, pin, and prove the VENDORED code is what loaded ----
# Kaggle reverts to its base image every session, so this runs every time.
import json
from src import setup_env

setup_env.install_dependencies()
setup_env.ensure_pycocotools_mask()
run = setup_env.bootstrap(RUN_MODE)

from src import manifest, train_utils

NUM_GPUS = NUM_GPUS if NUM_GPUS is not None else max(1, train_utils.visible_gpus())
lock = setup_env.write_requirements_lock()
environment = setup_env.env_report()
manifest.record_environment(environment)

# The repo vendors MODIFIED detectron2 / pycocotools (multi-label partial
# annotations, 3-tier category schema). A pip-installed copy silently shadows
# them and every number changes, so this is an assertion, not a warning.
found = setup_env.assert_vendored()
for module, path in found.items():
    print("{:14s} -> {}".format(module, path))
import detectron2, pycocotools, evaluator                              # noqa: F401
from hierarchialdet.util.coco_3class_eval import COCOEvaluator         # noqa: F401
from hierarchialdet.dataset_mapper_patched import DiffusionDetDatasetMapper  # noqa: F401
print("full import chain OK | commit {} | {} GPU process(es)".format(
    environment["repo_commit"][:12], NUM_GPUS))


In [ ]:
# ---- What can this session do, and what was trained? ----
from src import data_convert, degradations, eval_utils, figures, registration, tables

HAS_GPU = setup_env.gpu_report().get("available", False)
print("GPU available:", HAS_GPU, "->", "full pass" if HAS_GPU
      else "CPU pass (benchmark + asset build only)")

paths = data_convert.layout()
CFG = {name: os.path.join(setup_env.CONFIGS_REPRO, "diffdet.dentex.{}.yaml".format(name))
       for name in ("quadrant", "enumeration", "diagnosis", "base_diffusiondet")}
training = setup_env.read_notebook_summary("02_train_all") or {}
weights = training.get("weights") or {}
FULL_WEIGHTS = (weights.get("variants") or {}).get("full")

TIER_CLASSES = {0: 4, 1: 8, 2: 4}
models = {}
for label, key, config in (("Stage_quadrant", "quadrant", CFG["quadrant"]),
                           ("Stage_enumeration", "enumeration", CFG["enumeration"])):
    path = weights.get(key)
    if path and os.path.exists(path):
        models[label] = {"weights": path, "config": config, "tier": 2,
                         "overrides": [], "json_override": None}
for variant, path in (weights.get("variants") or {}).items():
    if os.path.exists(path):
        models[setup_env.VARIANT_LABELS[variant]] = {
            "weights": path, "config": CFG["diagnosis"], "tier": 2,
            "overrides": [], "json_override": None}
for tier_str, path in (weights.get("base") or {}).items():
    if os.path.exists(path):
        tier = int(tier_str)
        models["DiffusionDet_base_tier{}".format(tier)] = {
            "weights": path, "config": CFG["base_diffusiondet"], "tier": 0,
            "overrides": ["MODEL.DiffusionDet.NUM_CLASSES",
                          "[{}, 8, 4]".format(TIER_CLASSES[tier])],
            "json_override": data_convert.flat_json_path(tier, "test"),
            "flat_tier": tier}
for label, spec in models.items():
    print("{:26s} {}".format(label, spec["weights"]))
if HAS_GPU:
    assert models, "no trained checkpoints found — run notebook 02, or attach its dataset"


In [ ]:
# ---- Main evaluation: every model, every tier, three inference seeds ----
# One inference pass at tier 2 scores all three tiers, because the authors'
# inference_on_dataset stores the predictions once and evaluates tiers 0..k.
if HAS_GPU:
    for label, spec in models.items():
        print("\n=== {} ({} seeds) ===".format(label, len(run.eval_seeds)))
        payload = eval_utils.evaluate_multi_seed(
            spec["weights"], spec["config"], run.eval_seeds,
            split="diagnosis_test", tier=spec["tier"], overrides=spec["overrides"],
            limit=run.eval_limit, json_override=spec["json_override"],
            image_dir_override=paths["img_test"] if spec["json_override"] else None)
        payload["label"] = label
        payload["flat_tier"] = spec.get("flat_tier")
        eval_utils.save_result(manifest.result_name("eval_main", model=label), payload)
        for tier, cells in payload["aggregate"].items():
            ap = cells.get("AP", {})
            if ap.get("mean") is not None:
                print("  {:12s} AP {:.2f} ± {:.2f}   AR {}".format(
                    tier, ap["mean"], ap["std"],
                    "{:.3f}".format(cells["AR"]["mean"])
                    if cells["AR"]["mean"] is not None else "n/a"))


In [ ]:
# ---- Runtime, zero-detection and degenerate-box accounting ----
if HAS_GPU:
    for label, spec in models.items():
        output = os.path.join(setup_env.RUNS_DIR, "predictions", "{}_test.json".format(label))
        summary_i = eval_utils.dump_predictions(
            spec["weights"], spec["config"], "diagnosis_test", spec["tier"], output,
            seed=run.eval_seeds[0], limit=run.eval_limit, overrides=spec["overrides"],
            json_override=spec["json_override"],
            image_dir_override=paths["img_test"] if spec["json_override"] else None)
        eval_utils.save_result(
            manifest.result_name("runtime", model=label, device=summary_i["device"],
                                 step=summary_i["sample_step"]), summary_i)
        print("{:26s} {:.3f} s/image  {} images with no detections  failures {}".format(
            label, summary_i["mean_seconds_per_image"] or float("nan"),
            summary_i["images_with_no_detections"], summary_i["failure_counts"]))


In [ ]:
# ---- Checkpoint trajectory: is the ablation ordering stable over training? ----
# Each variant at ~1/3, ~2/3 and 1/1 of its budget. Stable ordering across
# checkpoints is evidence the ablation conclusion is not an artifact of the
# shortened schedule; unstable ordering is itself the finding.
if HAS_GPU:
    for record in training.get("records", []):
        if record.get("kind") != "diagnosis_variant":
            continue
        variant = record["variant"]
        max_iter = record.get("max_iter") or record.get("plan", {}).get("max_iter")
        points = []
        for iteration, name in sorted((record.get("trajectory") or {}).items(),
                                      key=lambda kv: int(kv[0])):
            checkpoint = os.path.join(train_utils.run_dir(record["name"]), name + ".pth")
            if not os.path.exists(checkpoint):
                print("missing trajectory checkpoint (skipped):", checkpoint)
                continue
            payload = eval_utils.evaluate_checkpoint(
                checkpoint, CFG["diagnosis"], split="diagnosis_test", tier=2,
                seed=run.eval_seeds[0], limit=run.eval_limit)
            points.append({"iteration": int(iteration), "checkpoint": name,
                           "progress": round(int(iteration) / max_iter, 3),
                           "result": payload})
            print("{:20s} @{:>6} iters  diagnosis AP {}".format(
                variant, iteration, payload["tiers"]["diagnosis"]["metrics"]["AP"]))
        final = eval_utils.load_result(
            manifest.result_name("eval_main", model=setup_env.VARIANT_LABELS[variant]))
        if final:
            points.append({"iteration": max_iter, "checkpoint": "model_final",
                           "progress": 1.0, "result": final["runs"][0],
                           "aggregate": final["aggregate"]})
        eval_utils.save_result(manifest.result_name("trajectory", variant=variant),
                               {"variant": variant, "max_iter": max_iter, "points": points})


In [ ]:
# ---- Robustness 1: diffusion sampling steps (accuracy AND measured latency) ----
if HAS_GPU and FULL_WEIGHTS:
    for step in run.step_sweep:
        payload = eval_utils.evaluate_multi_seed(
            FULL_WEIGHTS, CFG["diagnosis"], run.robustness_seeds,
            split="diagnosis_test", tier=2, sample_step=step, limit=run.eval_limit)
        eval_utils.save_result(manifest.result_name("steps", step=step), payload)
        latency = eval_utils.dump_predictions(
            FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2,
            os.path.join(setup_env.RUNS_DIR, "predictions", "full_step{}.json".format(step)),
            seed=run.robustness_seeds[0], sample_step=step, limit=run.eval_limit)
        eval_utils.save_result(manifest.result_name("steps_latency", step=step), latency)
        print("steps={:2d}  diagnosis AP {:.2f}  {:.3f} s/image".format(
            step, payload["aggregate"]["diagnosis"]["AP"]["mean"],
            latency["mean_seconds_per_image"]))


In [ ]:
# ---- Robustness 2: degradation grid ----
# Downscaled images are resampled back to the original resolution, so the
# ground-truth boxes stay valid and this measures lost detail, not a changed
# coordinate system.
if HAS_GPU and FULL_WEIGHTS:
    DEGRADED_ROOT = os.path.join(setup_env.RUNS_DIR, "degraded")
    for condition in degradations.degradation_grid(run):
        image_dir = paths["img_test"] if condition["kind"] == "none" else \
            degradations.build_degraded_images(paths["img_test"], condition["kind"],
                                               condition["severity"], DEGRADED_ROOT)
        payload = eval_utils.evaluate_multi_seed(
            FULL_WEIGHTS, CFG["diagnosis"], run.robustness_seeds,
            split="diagnosis_test", tier=2, limit=run.eval_limit,
            image_dir_override=image_dir)
        payload["condition"] = condition
        eval_utils.save_result(
            manifest.result_name("degradation", condition=condition["label"]), payload)
        print("{:20s} diagnosis AP {:.2f}".format(
            condition["label"], payload["aggregate"]["diagnosis"]["AP"]["mean"]))


In [ ]:
# ---- Robustness 3: clean vs stress subsets, for every trained model ----
if HAS_GPU:
    for which in ("clean", "stress"):
        subset_json = os.path.join(paths["subsets"], "test_{}.json".format(which))
        assert os.path.exists(subset_json), "run notebook 01 first: {}".format(subset_json)
        for label, spec in models.items():
            if spec["json_override"]:
                continue          # flat baselines have their own ground truth
            payload = eval_utils.evaluate_multi_seed(
                spec["weights"], spec["config"], run.robustness_seeds,
                split="diagnosis_test", tier=2, json_override=subset_json,
                limit=run.eval_limit)
            payload.update({"subset": which, "label": label})
            eval_utils.save_result(
                manifest.result_name("subset", model=label, subset=which), payload)
            print("{:24s} {:6s} diagnosis AP {:.2f}".format(
                label, which, payload["aggregate"]["diagnosis"]["AP"]["mean"]))


In [ ]:
# ---- Robustness 4: hierarchy fault injection ----
# The prior-tier boxes the diagnosis model consumes are deliberately corrupted.
# Inference-time injection is OFF for every other experiment (that reproduces the
# released behaviour, which never injects at inference), so the jitter=0/drop=0
# condition — not the main table — is this experiment's reference point.
if HAS_GPU and FULL_WEIGHTS:
    prior_test = (training.get("noisy_boxes") or {}).get("prior_over_test")
    assert prior_test and os.path.exists(prior_test), (
        "notebook 02 must produce the enumeration model's predictions over the test split")
    for condition in degradations.fault_grid(run):
        with eval_utils.noisy_box_inference(prior_test, jitter=condition["jitter"],
                                            drop=condition["drop"]):
            payload = eval_utils.evaluate_multi_seed(
                FULL_WEIGHTS, CFG["diagnosis"], run.robustness_seeds,
                split="diagnosis_test", tier=2, limit=run.eval_limit)
        payload["condition"] = condition
        eval_utils.save_result(
            manifest.result_name("fault", condition=condition["label"]), payload)
        print("{:16s} ({:s}) diagnosis AP {:.2f}".format(
            condition["label"], condition["axis"],
            payload["aggregate"]["diagnosis"]["AP"]["mean"]))
    setup_env.log_deviation(
        "inference-time prior-tier box injection used for the fault-injection experiment",
        "the released code injects prior-tier boxes only during training; the "
        "inference-time path exists in detector.py but is entirely commented out and "
        "referenced unpublished paths. It is off for every other experiment, so no "
        "main-table number depends on it",
        "03_evaluate_and_build_assets")


In [ ]:
# ---- Error analysis and the qualitative figures (need the images themselves) ----
if HAS_GPU and FULL_WEIGHTS:
    predictions_path = os.path.join(setup_env.RUNS_DIR, "predictions", "Ours_full_test.json")
    if not os.path.exists(predictions_path):
        eval_utils.dump_predictions(FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2,
                                    predictions_path, seed=0, limit=run.eval_limit)
    # Localisation and classification stay separate: a box that overlaps a real
    # tooth but carries the wrong diagnosis is a matched detection with a class
    # disagreement, not a miss plus a false positive.
    analysis = eval_utils.error_analysis(predictions_path, paths["test_diagnosis"], tier=2)
    eval_utils.save_result(manifest.result_name("errors", model="Ours_full"), analysis)
    print(json.dumps(analysis["totals"], indent=2))
    print("\nerror rate by tier (wrong label among correctly localised boxes):")
    for tier, rate in analysis["error_rate_by_tier"].items():
        print("  tier {}: {:.3f}".format(tier, rate))
    for tier, pairs in analysis["confusion"].items():
        print("  tier {} top confusions: {}".format(tier, list(pairs.items())[:5]))

    figure = figures.grouped_bars(
        ["quadrant", "enumeration", "diagnosis"],
        {"wrong label": [analysis["error_rate_by_tier"][str(t)] for t in range(3)]},
        "fraction of localised boxes with the wrong label",
        "Where errors cluster across the hierarchy",
        width=figures.SINGLE_COLUMN, rotate=0)
    figures.save_figure(figure, "error_clusters",
                        "Per-tier classification error rate among correctly localised "
                        "detections (IoU >= 0.5), full model, test split.",
                        "03_evaluate_and_build_assets", run.mode, "figure:error_clusters",
                        inputs=[predictions_path, paths["test_diagnosis"]])


In [ ]:
# ---- Failure gallery and curated overlays ----
if HAS_GPU and FULL_WEIGHTS:
    with open(paths["test_diagnosis"]) as handle:
        truth = json.load(handle)
    gt_by_image = {}
    for annotation in truth["annotations"]:
        gt_by_image.setdefault(annotation["image_id"], []).append(annotation)
    with open(predictions_path) as handle:
        predictions = json.load(handle)
    pred_by_image = {}
    for record in predictions:
        if record.get("score", 0) >= 0.5:
            pred_by_image.setdefault(record["image_id"], []).append(record)
    names = {level: {c["id"]: str(c["name"])
                     for c in truth["categories_{}".format(level + 1)]} for level in range(3)}
    by_id = {image["id"]: image for image in truth["images"]}

    def panel(image_id, title):
        gt = [(a["bbox"], names[2].get(a.get("category_id_3"), ""))
              for a in gt_by_image.get(image_id, [])]
        pred = [(r["bbox"], "{}({:.2f})".format(
            names[2].get(r.get("category_id_3"), r.get("category_id_3")), r.get("score", 0)))
            for r in pred_by_image.get(image_id, [])]
        return {"image_path": os.path.join(paths["img_test"], by_id[image_id]["file_name"]),
                "title": title, "gt": gt, "pred": pred}

    gallery = eval_utils.select_gallery_cases(analysis, per_category=2)
    gallery_ids, panels = {}, []
    for bucket, rows in gallery.items():
        gallery_ids[bucket] = [r["image_id"] for r in rows]
        for row in rows:
            panels.append(panel(row["image_id"], "{}\n{}".format(bucket, row["file_name"])))
    if panels:
        figure = figures.overlay_grid(panels, columns=4, panel_height=2.0,
                                      title="Failure gallery — blue solid: ground truth, "
                                            "orange dashed: prediction")
        figures.save_figure(figure, "failure_gallery",
                            "Representative failures of the full model on the DENTEX "
                            "test split, one row per failure mode.",
                            "03_evaluate_and_build_assets", run.mode, "figure:qualitative",
                            inputs=[predictions_path], note=json.dumps(gallery_ids))

    with open(os.path.join(paths["subsets"], "clean_stress.json")) as handle:
        subsets = json.load(handle)
    curated = subsets["clean"]["image_ids"][:6] + subsets["stress"]["image_ids"][:6]
    panels = [panel(i, "{} ({})".format(
        by_id[i]["file_name"],
        "clean" if i in subsets["clean"]["image_ids"] else "stress"))
        for i in curated if i in by_id]
    figure = figures.overlay_grid(panels, columns=4, panel_height=2.0,
                                  title="Prediction vs ground truth, 12 curated test images")
    figures.save_figure(figure, "qualitative_overlays",
                        "Full-model predictions (orange, dashed) against ground truth "
                        "(blue, solid) on six clean and six stress-subset test images.",
                        "03_evaluate_and_build_assets", run.mode, "figure:qualitative",
                        inputs=[predictions_path], note=json.dumps(curated))
    print(json.dumps(gallery_ids, indent=2))


In [ ]:
# ---- CPU or GPU inference cost, whichever this session can measure ----
CPU_IMAGES = 20
if FULL_WEIGHTS and HAS_GPU:
    for step in (1, 4):
        output = os.path.join(setup_env.RUNS_DIR, "predictions",
                              "full_gpu_step{}.json".format(step))
        measured = eval_utils.dump_predictions(
            FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2, output, seed=0,
            device="cuda", sample_step=step, limit=run.eval_limit)
        measured.update(eval_utils.model_size_report(FULL_WEIGHTS))
        eval_utils.save_result(
            manifest.result_name("lowresource", device="cuda", step=step), measured)
        print("step {}: {:.3f} s/image, {:.1f} img/min, peak VRAM {} MB, {} M params".format(
            step, measured["mean_seconds_per_image"], measured["images_per_minute"],
            measured.get("peak_gpu_memory_mb"), measured["parameters_millions"]))
elif FULL_WEIGHTS:
    output = os.path.join(setup_env.RUNS_DIR, "predictions", "full_cpu20.json")
    measured = eval_utils.dump_predictions(
        FULL_WEIGHTS, CFG["diagnosis"], "diagnosis_test", 2, output, seed=0,
        device="cpu", limit=CPU_IMAGES, sample_step=1)
    measured.update(eval_utils.model_size_report(FULL_WEIGHTS))
    eval_utils.save_result(manifest.result_name("lowresource", device="cpu"), measured)
    print("CPU: {:.2f} s/image over {} images".format(
        measured["mean_seconds_per_image"], measured["images"]))


In [ ]:
# ---- CPU vs GPU: same weights, same seed — do the detections agree? ----
agreement = None
cpu_predictions = os.path.join(setup_env.RUNS_DIR, "predictions", "full_cpu20.json")
gpu_predictions = os.path.join(setup_env.RUNS_DIR, "predictions", "full_gpu_step1.json")
if os.path.exists(cpu_predictions) and os.path.exists(gpu_predictions):
    with open(cpu_predictions) as handle:
        cpu_records = json.load(handle)
    with open(gpu_predictions) as handle:
        gpu_records = json.load(handle)
    shared = {r["image_id"] for r in cpu_records} & {r["image_id"] for r in gpu_records}
    matched_count = total = 0
    for image_id in shared:
        left = [r for r in cpu_records if r["image_id"] == image_id and r["score"] >= 0.5]
        right = [r for r in gpu_records if r["image_id"] == image_id and r["score"] >= 0.5]
        for record in left:
            total += 1
            if any(eval_utils._iou(record["bbox"], other["bbox"]) >= 0.5
                   and record.get("category_id_3") == other.get("category_id_3")
                   for other in right):
                matched_count += 1
    agreement = {"images_compared": len(shared), "cpu_detections": total,
                 "matched_on_gpu": matched_count,
                 "agreement_rate": round(matched_count / total, 4) if total else None,
                 "note": "residual disagreement is CPU/GPU kernel nondeterminism plus "
                         "the stochastic noisy-box start"}
    print(json.dumps(agreement, indent=2))
else:
    print("run this notebook once on GPU and once on CPU to get the agreement check")


In [ ]:
# ---- Load every raw result ----
def collect(kind, key):
    out = {}
    for name in manifest.list_results(kind):
        parts = manifest.parse_result_name(name)
        if parts["kind"] != kind:
            continue
        out[parts.get(key, name)] = eval_utils.load_result(name)
    return out

main_results = collect("eval_main", "model")
step_results = collect("steps", "step")
step_latency = collect("steps_latency", "step")
degradation_results = collect("degradation", "condition")
fault_results = collect("fault", "condition")
runtime_results = collect("runtime", "model")
lowresource = collect("lowresource", "device")
trajectories = collect("trajectory", "variant")
subset_results = {}
for name in manifest.list_results("subset"):
    parts = manifest.parse_result_name(name)
    subset_results["{}|{}".format(parts.get("model"), parts.get("subset"))] = \
        eval_utils.load_result(name)
print({k: len(v) for k, v in {
    "main": main_results, "steps": step_results, "degradation": degradation_results,
    "fault": fault_results, "subsets": subset_results, "runtime": runtime_results,
    "trajectory": trajectories, "lowresource": lowresource}.items()})


In [ ]:
# ---- Tables ----
NB = "03_evaluate_and_build_assets"
written = {}

if main_results:
    written["main_results"] = tables.write_table(
        "main_results", tables.main_results_rows(main_results),
        ["tier", "method"] + list(tables.METRIC_COLUMNS) + ["seeds"],
        "Per-tier detection metrics on the DENTEX test split, mean +/- std over "
        "{} inference seeds (RUN_MODE={}).".format(len(run.eval_seeds), run.mode),
        NB, run.mode, "table:main_results")
    written["original_vs_reproduced"] = tables.write_table(
        "original_vs_reproduced", tables.comparison_rows(main_results),
        ["tier", "method", "metric", "original", "reproduced", "reproduced_std",
         "delta", "delta_pct"],
        "Reproduced values against the originals reported in HierarchicalDet "
        "(MICCAI 2023, Table 1). 'not run' marks rows this run mode did not train.",
        NB, run.mode, "table:original_vs_reproduced",
        note="original column is quoted from the paper, not produced here")
    written["seed_variance"] = tables.write_table(
        "seed_variance", tables.seed_variance_rows(main_results),
        ["method", "tier", "metric", "n_seeds", "mean", "std", "min", "max", "values"],
        "Spread across inference seeds. DiffusionDet denoises from random boxes, so "
        "this is inherent inference variance at fixed weights.",
        NB, run.mode, "table:seed_variance")
    written["per_class_ap"] = tables.write_table(
        "per_class_ap", tables.per_class_rows(main_results),
        ["method", "tier", "class", "AP_mean", "n_seeds", "note"],
        "Per-class AP. OUR EXTENSION: the original paper reports tier-level "
        "aggregates only, so there is no reference column.",
        NB, run.mode, "table:main_results")
else:
    for asset_class in ("table:main_results", "table:original_vs_reproduced",
                        "table:seed_variance"):
        tables.record_not_run(asset_class, NB, run.mode, "no evaluation results found")

for name, payloads, axis, asset_class, caption in (
    ("step_sweep", step_results, "sampling_steps", "table:step_sweep",
     "Accuracy against diffusion sampling steps, full model."),
    ("degradation", degradation_results, "condition", "table:degradation",
     "Accuracy under image degradation, full model, test split."),
    ("clean_vs_stress", subset_results, "model_subset", "table:clean_vs_stress",
     "Clean versus stress evaluation subsets (selection rule in the dataset audit)."),
    ("fault_injection", fault_results, "condition", "table:fault_injection",
     "Diagnosis-tier accuracy when the prior-tier boxes feeding noisy-box "
     "manipulation are jittered or dropped."),
):
    if not payloads:
        tables.record_not_run(asset_class, NB, run.mode, "experiment not run in this mode")
        continue
    extra = None
    columns = [axis, "tier"] + list(tables.METRIC_COLUMNS)
    if name == "step_sweep" and step_latency:
        extra = {k: {"seconds_per_image":
                     (step_latency.get(k) or {}).get("mean_seconds_per_image")}
                 for k in payloads}
        columns.append("seconds_per_image")
    written[name] = tables.write_table(name, tables.sweep_rows(payloads, axis, extra),
                                       columns, caption, NB, run.mode, asset_class)

if runtime_results:
    written["failure_counts"] = tables.write_table(
        "failure_counts", tables.failure_rows(runtime_results),
        ["method", "images", "detections", "images_with_no_detections", "degenerate_box",
         "box_out_of_image", "box_covers_whole_image", "extreme_aspect_ratio", "crashes"],
        "Inference failure accounting on the test split.",
        NB, run.mode, "table:failure_counts")
else:
    tables.record_not_run("table:failure_counts", NB, run.mode, "no runtime results")

low_rows = [{"method": "Ours_full", **payload} for payload in lowresource.values() if payload]
if low_rows:
    written["low_resource"] = tables.write_table(
        "low_resource", tables.low_resource_rows(low_rows),
        ["method", "device", "sample_step", "seconds_per_image", "images_per_minute",
         "peak_gpu_memory_mb", "parameters_millions", "checkpoint_mb", "images"],
        "Low-resource benchmark: CPU and GPU inference cost of the full model.",
        NB, run.mode, "table:low_resource")
else:
    tables.record_not_run("table:low_resource", NB, run.mode, "no benchmark measured yet")


In [ ]:
# ---- Figures ----
if step_results:
    series = []
    ordered_steps = sorted(step_results, key=lambda s: int(s))
    for tier in tables.TIER_ORDER:
        points = [(float(s), (step_results[s]["aggregate"].get(tier, {}).get("AP", {}) or {}))
                  for s in ordered_steps]
        points = [(x, c["mean"], c.get("std") or 0.0) for x, c in points
                  if c.get("mean") is not None]
        if points:
            series.append({"label": tier, "x": [p[0] for p in points],
                           "y": [p[1] for p in points], "yerr": [p[2] for p in points]})
    latency = {int(k): (v or {}).get("mean_seconds_per_image")
               for k, v in step_latency.items() if v}
    figure = figures.ap_vs_steps_figure(series, latency or None)
    figures.save_figure(figure, "ap_vs_steps",
                        "AP against diffusion sampling steps for the full model, with "
                        "measured per-image latency on the right axis. Error bars are "
                        "the std over inference seeds.",
                        NB, run.mode, "figure:ap_vs_steps")
else:
    figures.record_not_run("figure:ap_vs_steps", NB, run.mode, "step sweep not run")

if degradation_results:
    baseline = ((degradation_results.get("clean") or {}).get("aggregate", {})
                .get("diagnosis", {}).get("AP", {}) or {}).get("mean")
    series = []
    for kind in degradations.KINDS:
        points = []
        for payload in degradation_results.values():
            condition = payload.get("condition") or {}
            if condition.get("kind") != kind:
                continue
            cell = payload["aggregate"]["diagnosis"]["AP"]
            points.append((float(condition["severity"]), cell["mean"], cell.get("std") or 0.0))
        if points:
            points.sort()
            series.append({"label": kind, "x": [p[0] for p in points],
                           "y": [p[1] for p in points], "yerr": [p[2] for p in points]})
    figure = figures.sweep_figure(series, "severity (blur sigma / JPEG quality / scale)",
                                  "diagnosis AP [0.5:0.95]",
                                  "Robustness to image degradation",
                                  width=figures.DOUBLE_COLUMN * 0.6)
    figures.save_figure(figure, "degradation_curves",
                        "Diagnosis-tier AP under Gaussian blur, JPEG recompression and "
                        "downscaling. Severity axes are per-type and not comparable "
                        "across lines; the clean baseline is AP={}.".format(baseline),
                        NB, run.mode, "figure:degradation")
else:
    figures.record_not_run("figure:degradation", NB, run.mode, "degradation grid not run")

if fault_results:
    series = []
    for axis in ("jitter", "drop"):
        points = []
        for payload in fault_results.values():
            condition = payload.get("condition") or {}
            if condition.get("axis") != axis:
                continue
            cell = payload["aggregate"]["diagnosis"]["AP"]
            points.append((condition.get(axis, 0.0), cell["mean"], cell.get("std") or 0.0))
        if points:
            points.sort()
            series.append({"label": "prior-tier {}".format(axis),
                           "x": [p[0] for p in points], "y": [p[1] for p in points],
                           "yerr": [p[2] for p in points]})
    figure = figures.sweep_figure(series, "perturbation magnitude",
                                  "diagnosis AP [0.5:0.95]",
                                  "Sensitivity to an imperfect prior tier")
    figures.save_figure(figure, "fault_injection",
                        "Diagnosis-tier AP as the enumeration model's detections are "
                        "jittered (normalized-coordinate sigma) or randomly dropped "
                        "before being used as noisy boxes.",
                        NB, run.mode, "figure:fault_injection")
else:
    figures.record_not_run("figure:fault_injection", NB, run.mode, "fault injection not run")


In [ ]:
# ---- Per-class bars and the checkpoint-trajectory figure ----
per_class = [r for r in tables.per_class_rows(main_results)
             if r["tier"] in ("enumeration", "diagnosis")]
if per_class:
    for tier in ("enumeration", "diagnosis"):
        rows = [r for r in per_class if r["tier"] == tier]
        if not rows:
            continue
        classes = sorted({r["class"] for r in rows})
        groups = {method: [next((r["AP_mean"] for r in rows
                                 if r["method"] == method and r["class"] == klass), 0.0)
                           for klass in classes]
                  for method in sorted({r["method"] for r in rows})}
        figure = figures.grouped_bars(classes, groups, "AP [0.5:0.95]",
                                      "Per-class AP — {} tier (our extension)".format(tier),
                                      rotate=45 if tier == "diagnosis" else 90)
        figures.save_figure(figure, "per_class_ap_{}".format(tier),
                            "Per-class AP at the {} tier. OUR EXTENSION: the original "
                            "paper reports tier-level aggregates only.".format(tier),
                            NB, run.mode, "figure:per_class_ap")
else:
    figures.record_not_run("figure:per_class_ap", NB, run.mode, "no evaluation results")

ordering, stable = {}, {}
if trajectories:
    shaped = {}
    for variant, payload in trajectories.items():
        per_tier = {}
        for point in payload.get("points", []):
            aggregate = point.get("aggregate")
            tiers = (point.get("result") or {}).get("tiers", {})
            for tier in tables.TIER_ORDER:
                if aggregate:
                    cell = aggregate.get(tier, {}).get("AP", {})
                    value, spread = cell.get("mean"), cell.get("std")
                else:
                    value, spread = tiers.get(tier, {}).get("metrics", {}).get("AP"), 0.0
                if value is not None:
                    per_tier.setdefault(tier, []).append(
                        {"progress": point["progress"], "AP": value, "AP_std": spread})
        for tier in per_tier:
            per_tier[tier].sort(key=lambda p: p["progress"])
        shaped[setup_env.VARIANT_LABELS.get(variant, variant)] = per_tier
    figure = figures.trajectory_figure(shaped, list(tables.TIER_ORDER))
    figures.save_figure(figure, "checkpoint_trajectory",
                        "Per-tier AP against training progress for each diagnosis "
                        "variant. Stable ordering across checkpoints is evidence the "
                        "ablation conclusion is not an artifact of the shortened "
                        "schedule; unstable ordering is itself a finding.",
                        NB, run.mode, "figure:checkpoint_trajectory")
    for tier in tables.TIER_ORDER:
        by_progress = {}
        for variant, per_tier in shaped.items():
            for point in per_tier.get(tier, []):
                by_progress.setdefault(point["progress"], []).append((variant, point["AP"]))
        ordering[tier] = {str(p): [v for v, _ in sorted(vals, key=lambda kv: -kv[1])]
                          for p, vals in sorted(by_progress.items())}
    stable = {tier: len({tuple(order) for order in per_progress.values()}) == 1
              for tier, per_progress in ordering.items()}
    print(json.dumps({"ordering": ordering, "ordering_stable": stable}, indent=2))
else:
    figures.record_not_run("figure:checkpoint_trajectory", NB, run.mode,
                           "no trajectory checkpoints evaluated")


In [ ]:
# ---- repro_checklist.md ----
environment = manifest.load_manifest().get("environment", {})
data_summary = setup_env.read_notebook_summary("01_setup_and_data") or {}
records = [r for r in (training.get("records") or []) if r]

lines = ["# Reproducibility checklist", "",
         "Generated from executed-run records only.", "",
         "## Environment", "",
         "- repo commit: `{}`".format(environment.get("repo_commit")),
         "- python: {}".format(environment.get("python")),
         "- platform: {}".format(environment.get("platform")),
         "- GPU: {}".format(json.dumps(environment.get("gpu", {}).get("devices", []))),
         "- CUDA / cuDNN: {} / {}".format(environment.get("gpu", {}).get("cuda"),
                                          environment.get("gpu", {}).get("cudnn")),
         "- run mode: **{}**".format(run.mode),
         "- training seed: {} (the repo's own SEED)".format(setup_env.BASE_SEED),
         "- inference seeds: {}".format(list(run.eval_seeds)),
         "- multi-GPU: {}".format(json.dumps(training.get("multi_gpu", {}))), "",
         "## Converted dataset", ""]
for name, digest in (data_summary.get("dataset_hashes") or {}).items():
    lines.append("- `{}`: `{}`".format(name, digest))

lines += ["", "## Runs", "",
          "| run | config hash | iterations | seed | batch | wall (s) | GPUs | stopped on budget |",
          "|---|---|---|---|---|---|---|---|"]
for record in records:
    lines.append("| {} | `{}` | {} | {} | {} | {} | {} | {} |".format(
        record.get("name"), (record.get("config_hash") or "n/a")[:12],
        record.get("max_iter"), record.get("seed"), record.get("ims_per_batch"),
        record.get("wall_seconds"), record.get("num_gpus"),
        record.get("stopped_on_time_budget")))

lines += ["", "## Exact commands", ""]
for record in records:
    command = (record.get("launch") or {}).get("command")
    if command:
        lines.append("```\n{}\n```".format(" ".join(str(c) for c in command)))

lines += ["", "## Remaining nondeterminism", "",
          "- mixed-precision (AMP) reduction order during training",
          "- atomics in the torchvision ROIAlign / NMS CUDA kernels",
          "- DataLoader worker interleaving (NUM_WORKERS=2)",
          "- inference starts from random noisy boxes, which is why every reported "
          "number is a mean over {} inference seeds".format(len(run.eval_seeds)), ""]

checklist = os.path.join(setup_env.PAPER_ASSETS, "repro_checklist.md")
with open(checklist, "w") as handle:
    handle.write("\n".join(lines) + "\n")
manifest.record_asset(checklist, "doc:repro_checklist", NB, run.mode)
manifest.record_asset(setup_env.DEVIATIONS_MD, "doc:deviations", NB, run.mode)
print("\n".join(lines[:35]))


In [ ]:
# ---- scope_and_claims.md: the claim-coverage matrix ----
tested_models = set(main_results)
coverage = []
for claim in tables.PAPER_CLAIMS:
    status, evidence = claim["default_status"], claim["evidence"]
    if claim.get("requires_models"):
        missing = [m for m in claim["requires_models"] if m not in tested_models]
        if missing:
            status = "cited, untested"
            evidence = "not trained in RUN_MODE={} (missing: {})".format(
                run.mode, ", ".join(missing))
    coverage.append({**claim, "status": status, "evidence": evidence})

gpu_hours = setup_env.gpu_hours_spent()
lines = ["# Scope, claims and what a reviewer can re-verify", "",
         "Run mode: **{}**. Measured GPU-hours actually spent: **{:.2f}**."
         .format(run.mode, gpu_hours), "",
         "## Claim coverage", "",
         "| # | Claim in the original paper | Status | Evidence |", "|---|---|---|---|"]
for index, claim in enumerate(coverage, 1):
    lines.append("| {} | {} | {} | {} |".format(
        index, claim["claim"], claim["status"], claim["evidence"]))

lines += ["", "## Compute actually spent", "", "| run | wall (s) | GPUs |", "|---|---|---|"]
for record in records:
    lines.append("| {} | {} | {} |".format(
        record.get("name"), record.get("wall_seconds"), record.get("num_gpus")))

lines += ["", "## Re-verifying without retraining", "",
          "Attach `{}` and `{}`, open `notebooks/03_evaluate_and_build_assets.ipynb` "
          "with Accelerator = GPU T4, set `RUN_MODE = \"{}\"`, and Run All."
          .format(CKPT_DATASET_SLUG, DATA_DATASET_SLUG, run.mode), "",
          "That reproduces every number in `tables/main_results.csv` from the released "
          "checkpoints, with no training. Re-running the same notebook on a CPU session "
          "rebuilds every table and figure for free.", "",
          "## Not a clinical claim", "",
          "Every artifact here is a research artifact produced under a constrained "
          "compute budget. None of it is validated for, or intended for, clinical use.",
          ""]
scope = os.path.join(setup_env.PAPER_ASSETS, "scope_and_claims.md")
with open(scope, "w") as handle:
    handle.write("\n".join(lines) + "\n")
manifest.record_asset(scope, "doc:scope_and_claims", NB, run.mode)
print("\n".join(lines[:30]))


In [ ]:
# ---- Contract check: nothing may be cited that is not in the manifest ----
contract = manifest.assert_asset_classes()
print(json.dumps(contract, indent=2))
print()
for row in manifest.summary_table():
    print("{:34s} {:8s} {}".format(row["asset_class"], row["status"], row["path"]))


In [ ]:
# ---- Notebook summary (the only cross-notebook contract) ----
summary = {
    "run_mode": run.mode,
    "had_gpu": HAS_GPU,
    "eval_seeds": list(run.eval_seeds),
    "models_evaluated": sorted(main_results),
    "aggregate": {k: v.get("aggregate") for k, v in main_results.items()},
    "cpu_gpu_agreement": agreement,
    "tables": written,
    "contract": contract,
    "ablation_ordering": ordering,
    "ablation_ordering_stable": stable,
    "gpu_hours_spent": gpu_hours,
    "manifest": manifest.MANIFEST_PATH,
}

path = setup_env.write_notebook_summary("03_evaluate_and_build_assets", summary)
print("wrote", path)
print(json.dumps(summary, indent=2, default=str)[:4000])


In [ ]:
# ---- Publish, last ----
# Round-trips paper_assets/ back into the checkpoint dataset so the next
# session (GPU or CPU) starts with every result this one produced.
publish = {"status": "disabled"}
if PUBLISH_KAGGLE_DATASET:
    publish = train_utils.publish_kaggle_dataset(
        CKPT_DATASET_SLUG, [setup_env.RUNS_DIR, setup_env.PAPER_ASSETS],
        "evaluation + paper assets ({} mode)".format(run.mode))
    summary["kaggle_publish"] = {k: v for k, v in publish.items()
                                 if k not in ("stdout", "stderr")}
    setup_env.write_notebook_summary("03_evaluate_and_build_assets", summary)
print(json.dumps({k: v for k, v in publish.items() if k not in ("stdout", "stderr")},
                 indent=2))
print("\npaper_assets/ is at:", setup_env.PAPER_ASSETS)
